In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/network_telemetry_anomaly.csv"
)

df["timestamp"] = pd.to_datetime(
    df["timestamp"]
)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nAnomaly distribution:")
print(df["anomaly_type"].value_counts(dropna=False))

Shape: (20160, 9)

Columns:
['timestamp', 'device_id', 'bandwidth', 'latency', 'packet_loss', 'cpu_usage', 'memory_usage', 'is_anomaly', 'anomaly_type']

Anomaly distribution:
anomaly_type
normal                 20139
traffic_spike              7
network_degradation        7
cpu_overload               7
Name: count, dtype: int64


In [10]:
anomalies = df[df["is_anomaly"] == 1].copy()

print("Jumlah anomaly:", len(anomalies))

print("\nDetail anomaly:")
display(
    anomalies[
        [
            "timestamp",
            "device_id",
            "bandwidth",
            "latency",
            "packet_loss",
            "cpu_usage",
            "memory_usage",
            "anomaly_type"
        ]
    ].sort_values("timestamp")
)

Jumlah anomaly: 21

Detail anomaly:


,timestamp,device_id,bandwidth,latency,packet_loss,cpu_usage,memory_usage,anomaly_type
10884,2026-09-03 19:00:00,R-006,11.818250,106.974693,4.184829,30.750000,44.39,network_degradation
10885,2026-09-03 19:05:00,R-006,11.834045,114.607232,3.092901,45.780000,51.22,network_degradation
10886,2026-09-03 19:10:00,R-006,13.042422,118.318092,4.215090,28.530000,47.93,network_degradation
10887,2026-09-03 19:15:00,R-006,15.247564,122.803499,3.341048,31.120000,47.22,network_degradation
10888,2026-09-03 19:20:00,R-006,14.319450,139.258798,3.130103,32.840000,53.85,network_degradation
10889,2026-09-03 19:25:00,R-006,12.912291,109.983689,4.897771,46.660000,47.27,network_degradation
10890,2026-09-03 19:30:00,R-006,16.118529,125.711722,4.931264,40.690000,52.32,network_degradation
17112,2026-09-04 10:00:00,R-009,45.830000,35.000000,0.440000,93.370861,53.34,cpu_overload
17113,2026-09-04 10:05:00,R-009,45.160000,33.260000,0.200000,98.556429,56.11,cpu_overload
17114,2026-09-04 10:10:00,R-009,45.960000,32.170000,0.320000,96.587945,57.72,cpu_overload


In [11]:
incident_summary = (
    anomalies
    .groupby(["device_id", "anomaly_type"])
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        event_count=("timestamp", "count"),
        avg_bandwidth=("bandwidth", "mean"),
        avg_latency=("latency", "mean"),
        avg_packet_loss=("packet_loss", "mean"),
        avg_cpu_usage=("cpu_usage", "mean"),
        avg_memory_usage=("memory_usage", "mean")
    )
    .reset_index()
)

incident_summary.insert(
    0,
    "incident_id",
    [f"INC-{i:05d}" for i in range(1, len(incident_summary) + 1)]
)

display(incident_summary)

,incident_id,device_id,anomaly_type,start_time,end_time,event_count,avg_bandwidth,avg_latency,avg_packet_loss,avg_cpu_usage,avg_memory_usage
0,INC-00001,R-004,traffic_spike,2026-09-06 14:00:00,2026-09-06 14:30:00,7,96.015200,29.567143,0.378571,34.828571,42.427143
1,INC-00002,R-006,network_degradation,2026-09-03 19:00:00,2026-09-03 19:30:00,7,13.613222,119.665389,3.970429,36.624286,49.171429
2,INC-00003,R-009,cpu_overload,2026-09-04 10:00:00,2026-09-04 10:30:00,7,47.208571,34.892857,0.257143,93.890576,56.351429


In [12]:
def determine_severity(row):

    if (
        row["avg_latency"] >= 100
        or row["avg_packet_loss"] >= 3
    ):
        return "CRITICAL"

    elif (
        row["avg_cpu_usage"] >= 90
        or row["avg_bandwidth"] >= 90
    ):
        return "HIGH"

    else:
        return "MEDIUM"


incident_summary["severity"] = incident_summary.apply(
    determine_severity,
    axis=1
)

display(
    incident_summary[
        [
            "incident_id",
            "device_id",
            "anomaly_type",
            "severity",
            "start_time",
            "end_time"
        ]
    ]
)

,incident_id,device_id,anomaly_type,severity,start_time,end_time
0,INC-00001,R-004,traffic_spike,HIGH,2026-09-06 14:00:00,2026-09-06 14:30:00
1,INC-00002,R-006,network_degradation,CRITICAL,2026-09-03 19:00:00,2026-09-03 19:30:00
2,INC-00003,R-009,cpu_overload,HIGH,2026-09-04 10:00:00,2026-09-04 10:30:00


In [13]:
def determine_root_cause(row):

    if (
        row["avg_cpu_usage"] >= 90
        and row["avg_bandwidth"] < 80
    ):
        return "CPU Overload"

    elif (
        row["avg_bandwidth"] >= 90
        and row["avg_cpu_usage"] < 80
    ):
        return "Traffic Spike"

    elif (
        row["avg_latency"] >= 100
        and row["avg_packet_loss"] >= 3
        and row["avg_bandwidth"] < 30
    ):
        return "Network Degradation"

    else:
        return "Insufficient Evidence"


incident_summary["root_cause"] = incident_summary.apply(
    determine_root_cause,
    axis=1
)

display(
    incident_summary[
        [
            "incident_id",
            "device_id",
            "anomaly_type",
            "severity",
            "root_cause"
        ]
    ]
)

,incident_id,device_id,anomaly_type,severity,root_cause
0,INC-00001,R-004,traffic_spike,HIGH,Traffic Spike
1,INC-00002,R-006,network_degradation,CRITICAL,Network Degradation
2,INC-00003,R-009,cpu_overload,HIGH,CPU Overload


In [14]:
def generate_evidence(row):

    if row["root_cause"] == "CPU Overload":
        return (
            f"CPU usage averaged {row['avg_cpu_usage']:.2f}% "
            f"over {row['event_count']} telemetry events, "
            f"while bandwidth averaged {row['avg_bandwidth']:.2f}."
        )

    elif row["root_cause"] == "Traffic Spike":
        return (
            f"Bandwidth averaged {row['avg_bandwidth']:.2f} "
            f"over {row['event_count']} telemetry events, "
            f"while CPU usage averaged {row['avg_cpu_usage']:.2f}%."
        )

    elif row["root_cause"] == "Network Degradation":
        return (
            f"Latency averaged {row['avg_latency']:.2f} ms "
            f"and packet loss averaged {row['avg_packet_loss']:.2f}%, "
            f"while bandwidth averaged {row['avg_bandwidth']:.2f}."
        )

    else:
        return "Insufficient telemetry evidence."


incident_summary["evidence"] = incident_summary.apply(
    generate_evidence,
    axis=1
)

display(
    incident_summary[
        [
            "incident_id",
            "device_id",
            "severity",
            "root_cause",
            "evidence"
        ]
    ]
)

,incident_id,device_id,severity,root_cause,evidence
0,INC-00001,R-004,HIGH,Traffic Spike,Bandwidth averaged 96.02 over 7 telemetry even...
1,INC-00002,R-006,CRITICAL,Network Degradation,Latency averaged 119.67 ms and packet loss ave...
2,INC-00003,R-009,HIGH,CPU Overload,CPU usage averaged 93.89% over 7 telemetry eve...


In [24]:
def build_ai_prompt(row):

    return f"""
You are an AI network troubleshooting assistant.

Analyze the network incident using ONLY the telemetry and evidence provided below.

IMPORTANT RULES:
- Do not invent facts, measurements, causes, devices, applications, or events.
- Treat the provided root cause as a telemetry-based hypothesis, not absolute truth.
- Clearly distinguish observed evidence from your interpretation.
- If the evidence is insufficient to determine a cause, explicitly say so.
- Recommendations must be safe diagnostic steps.
- Do not recommend destructive actions or configuration changes without verification.

INCIDENT INFORMATION

Incident ID: {row["incident_id"]}
Device: {row["device_id"]}
Severity: {row["severity"]}

Time:
Start: {row["start_time"]}
End: {row["end_time"]}

Detected anomaly:
{row["anomaly_type"]}

Telemetry-based root cause:
{row["root_cause"]}

AVERAGE TELEMETRY

Bandwidth: {row["avg_bandwidth"]:.2f}
Latency: {row["avg_latency"]:.2f} ms
Packet loss: {row["avg_packet_loss"]:.2f}%
CPU usage: {row["avg_cpu_usage"]:.2f}%
Memory usage: {row["avg_memory_usage"]:.2f}%

EVIDENCE

{row["evidence"]}

TASK

Provide your analysis using exactly these sections:

1. Summary
Briefly explain what happened.

2. Evidence
Explain which telemetry values support the analysis.

3. Likely Cause
Explain the most likely cause based only on the available evidence.
If evidence is insufficient, state that clearly.

4. Potential Impact
Explain the possible impact on network performance.

5. Diagnostic Recommendations
Provide 3 safe diagnostic steps that a network administrator could perform.

6. Evidence Limitation
Explain what information is missing or what cannot be concluded from the available telemetry.

Keep the explanation concise, technical, and suitable for a network operations team.
""".strip()


incident_summary["ai_prompt"] = incident_summary.apply(
    build_ai_prompt,
    axis=1
)

In [16]:
from getpass import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass("Masukkan OpenAI API key: ")

print("API key berhasil dimuat ke Jupyter.")

Masukkan OpenAI API key:  ········


API key berhasil dimuat ke Jupyter.


In [17]:
from openai import OpenAI

client = OpenAI()

print("OpenAI client berhasil dibuat.")

OpenAI client berhasil dibuat.


In [18]:
import os

print("API key tersedia di Jupyter:", bool(os.getenv("OPENAI_API_KEY")))

API key tersedia di Jupyter: True


In [5]:
print(
    "API key tersedia di Jupyter:",
    bool(os.getenv("OPENAI_API_KEY"))
)

API key tersedia di Jupyter: True


In [19]:
response = client.responses.create(
    model="gpt-5.6-luna",
    input=incident_summary.loc[0, "ai_prompt"]
)

print(response.output_text)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [20]:
from getpass import getpass
import os

os.environ["GEMINI_API_KEY"] = getpass("Masukkan Gemini API key: ")

print("Gemini API key berhasil dimuat.")

Masukkan Gemini API key:  ········


Gemini API key berhasil dimuat.


In [21]:
from google import genai

gemini_client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

print("Gemini client berhasil dibuat.")

Gemini client berhasil dibuat.


In [23]:
interaction = gemini_client.interactions.create(
    model="gemini-3.8-flash",
    input="Balas singkat: koneksi Gemini berhasil."
)

print(interaction.output_text)

Koneksi Gemini berhasil.


In [25]:
print(incident_summary.loc[0, "ai_prompt"])

You are an AI network troubleshooting assistant.

Analyze the network incident using ONLY the telemetry and evidence provided below.

IMPORTANT RULES:
- Do not invent facts, measurements, causes, devices, applications, or events.
- Treat the provided root cause as a telemetry-based hypothesis, not absolute truth.
- Clearly distinguish observed evidence from your interpretation.
- If the evidence is insufficient to determine a cause, explicitly say so.
- Recommendations must be safe diagnostic steps.
- Do not recommend destructive actions or configuration changes without verification.

INCIDENT INFORMATION

Incident ID: INC-00001
Device: R-004
Severity: HIGH

Time:
Start: 2026-09-06 14:00:00
End: 2026-09-06 14:30:00

Detected anomaly:
traffic_spike

Telemetry-based root cause:
Traffic Spike

AVERAGE TELEMETRY

Bandwidth: 96.02
Latency: 29.57 ms
Packet loss: 0.38%
CPU usage: 34.83%
Memory usage: 42.43%

EVIDENCE

Bandwidth averaged 96.02 over 7 telemetry events, while CPU usage averaged 

In [28]:
prompt = incident_summary.loc[2, "ai_prompt"]

interaction = gemini_client.interactions.create(
    model="gemini-3.8-flash",
    input=prompt
)

print(interaction.output_text)

### 1. Summary
Between 2026-09-04 10:00:00 and 10:30:00, device R-009 triggered a HIGH severity anomaly alert for `cpu_overload`. Telemetry captured sustained high processor utilization averaging 93.89% across the 30-minute window.

### 2. Evidence
* **CPU Usage:** Averaged 93.89% over 7 telemetry events.
* **Memory Usage:** Averaged 56.35%.
* **Latency:** Averaged 34.89 ms.
* **Packet Loss:** Averaged 0.26%.
* **Bandwidth:** Averaged 47.21.

### 3. Likely Cause
Based on the available metrics, the device experienced severe CPU exhaustion, aligning with the hypothesized root cause of CPU Overload. However, the evidence is insufficient to identify the underlying trigger for the high CPU utilization (e.g., whether it was caused by control plane processing, software routing, an internal process malfunction, or external traffic demands).

### 4. Potential Impact
* **Control Plane Degradation:** High CPU usage can lead to delayed processing of routing updates, protocol keepalives, and manage

In [30]:
def analyze_incident_with_gemini(prompt):
    interaction = gemini_client.interactions.create(
        model="gemini-3.8-flash",
        input=prompt
    )

    return interaction.output_text

In [31]:
test_analysis = analyze_incident_with_gemini(
    incident_summary.loc[0, "ai_prompt"]
)

print(test_analysis)

### 1. Summary
Between 2026-09-06 14:00:00 and 14:30:00, router **R-004** experienced a HIGH severity incident characterized by a `traffic_spike` anomaly. During this 30-minute window, bandwidth utilization averaged 96.02 across 7 telemetry events, accompanied by low-level packet loss (0.38%) and moderate latency (29.57 ms), while device control-plane utilization (CPU at 34.83% and Memory at 42.43%) remained stable.

---

### 2. Evidence
* **Observed Metrics:**
  * **Bandwidth:** Averaged 96.02 across 7 telemetry events.
  * **Packet Loss:** Averaged 0.38%.
  * **Latency:** Averaged 29.57 ms.
  * **CPU Usage:** Averaged 34.83%.
  * **Memory Usage:** Averaged 42.43%.
* **Interpretation:** 
  The primary abnormal indicator is the high bandwidth reading (96.02), which coincides with the system-detected `traffic_spike`. The relatively low CPU and memory metrics demonstrate that the device itself was not under control-plane resource exhaustion, and packet forwarding was occurring primarily 

In [32]:
incident_summary["ai_analysis"] = incident_summary["ai_prompt"].apply(
    analyze_incident_with_gemini
)

print("AI analysis berhasil dibuat untuk semua incident.")

AI analysis berhasil dibuat untuk semua incident.


In [33]:
display(
    incident_summary[
        [
            "incident_id",
            "device_id",
            "severity",
            "root_cause",
            "ai_analysis"
        ]
    ]
)

,incident_id,device_id,severity,root_cause,ai_analysis
0,INC-00001,R-004,HIGH,Traffic Spike,### 1. Summary\nBetween 2026-09-06 14:00:00 an...
1,INC-00002,R-006,CRITICAL,Network Degradation,### 1. Summary\nBetween 2026-09-03 19:00:00 an...
2,INC-00003,R-009,HIGH,CPU Overload,### 1. Summary\nBetween 10:00:00 and 10:30:00 ...


In [34]:
incident_result = incident_summary[
    [
        "incident_id",
        "device_id",
        "severity",
        "anomaly_type",
        "start_time",
        "end_time",
        "root_cause",
        "evidence",
        "ai_analysis"
    ]
].copy()

display(incident_result)

,incident_id,device_id,severity,anomaly_type,start_time,end_time,root_cause,evidence,ai_analysis
0,INC-00001,R-004,HIGH,traffic_spike,2026-09-06 14:00:00,2026-09-06 14:30:00,Traffic Spike,Bandwidth averaged 96.02 over 7 telemetry even...,### 1. Summary\nBetween 2026-09-06 14:00:00 an...
1,INC-00002,R-006,CRITICAL,network_degradation,2026-09-03 19:00:00,2026-09-03 19:30:00,Network Degradation,Latency averaged 119.67 ms and packet loss ave...,### 1. Summary\nBetween 2026-09-03 19:00:00 an...
2,INC-00003,R-009,HIGH,cpu_overload,2026-09-04 10:00:00,2026-09-04 10:30:00,CPU Overload,CPU usage averaged 93.89% over 7 telemetry eve...,### 1. Summary\nBetween 10:00:00 and 10:30:00 ...


In [35]:
incident_result.to_json(
    "../data/incident_result.json",
    orient="records",
    date_format="iso",
    indent=2
)

print("incident_result.json berhasil disimpan.")

incident_result.json berhasil disimpan.


In [36]:
import json

with open("../data/incident_result.json", "r", encoding="utf-8") as f:
    incident_json = json.load(f)

print("Jumlah incident:", len(incident_json))
print("\nIncident ID:")

for incident in incident_json:
    print("-", incident["incident_id"])

Jumlah incident: 3

Incident ID:
- INC-00001
- INC-00002
- INC-00003
